# ADBench baselines on the same splits as STRAP

Reruns the 14 unsupervised baselines of ADBench Table D4 (IForest, CBLOF, ECOD, COPOD, PCA, HBOS, KNN, OCSVM, SOD, LOF, LODA, COF, DeepSVDD, DAGMM) with default hyperparameters, as ADBench does, on **exactly the splits used by the STRAP notebook**:
ADBench's data pipeline (cap at 10,000 samples, stratified 70/30 split, seeds 1, 2, 3, MinMax scaling), verified identical to ADBench's `DataGenerator`, **except** that ADBench's resampling of datasets under 1,000 samples is switched off: it draws with replacement before splitting, which puts copies of the same sample in both the training and the test part.

Two protocols, matching the STRAP notebook:
- `adbench`: fit on the full unlabeled training split (contains anomalies), as in Table D4.
- `clean_holdout`: fit on the normal samples of the same training split.

For every (protocol, dataset, seed) the notebook saves the test labels and, per method, the training scores and the test scores. Training scores are needed for label-free thresholds (e.g. PyOD's default 10% contamination rule, or the (1−α) quantile).

Note: in PyOD, SOD and COF score the test set on its own (the fitted training data is not used at scoring time), so their results do not depend on the protocol.

**Runtime:** the 12 classical detectors run on CPU (about 3 hours for everything; SOD and COF dominate). DeepSVDD and DAGMM use the GPU if available. Results save after each (protocol, dataset, seed), so the notebook resumes after a disconnect.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
!pip install -q pyod wget
import os, sys, json, time, random, inspect, subprocess, warnings, urllib.request
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings('ignore')

# ADBench's own DAGMM implementation: sparse clone (code only, no datasets)
if not os.path.exists('/content/ADBench'):
    subprocess.run('git clone -q --depth 1 --filter=blob:none --sparse https://github.com/Minqi824/ADBench.git /content/ADBench', shell=True, check=True)
    subprocess.run('cd /content/ADBench && git sparse-checkout set adbench/baseline/DAGMM', shell=True, check=True)
    subprocess.run('cd /content/ADBench && git checkout -q', shell=True, check=True)
sys.path.insert(0, '/content/ADBench')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device for deep baselines:', DEVICE)

In [ ]:
# ── Drive and data ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/STRAP_ADBench'
OUT_DIR    = os.path.join(DRIVE_ROOT, 'baseline_scores')
DATA_DIR   = '/content/adbench_data'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)

ADBENCH_47 = ['1_ALOI','2_annthyroid','3_backdoor','4_breastw','5_campaign','6_cardio',
  '7_Cardiotocography','8_celeba','9_census','10_cover','11_donors','12_fault','13_fraud',
  '14_glass','15_Hepatitis','16_http','17_InternetAds','18_Ionosphere','19_landsat',
  '20_letter','21_Lymphography','22_magic.gamma','23_mammography','24_mnist','25_musk',
  '26_optdigits','27_PageBlocks','28_pendigits','29_Pima','30_satellite','31_satimage-2',
  '32_shuttle','33_skin','34_smtp','35_SpamBase','36_speech','37_Stamps','38_thyroid',
  '39_vertebral','40_vowels','41_Waveform','42_WBC','43_WDBC','44_Wilt','45_wine',
  '46_WPBC','47_yeast']
RAW = 'https://raw.githubusercontent.com/Minqi824/ADBench/main/adbench/datasets/Classical/'
for name in ADBENCH_47:
    dst = os.path.join(DATA_DIR, name + '.npz')
    if not os.path.exists(dst): urllib.request.urlretrieve(RAW + name + '.npz', dst)
print(len(os.listdir(DATA_DIR)), 'datasets ready')

In [ ]:
# ── Splits (identical to the STRAP notebook) and baselines ─────────────
SEEDS = [1, 2, 3]
PROTOCOLS = ['adbench', 'clean_holdout']
RESAMPLE_SMALL = False   # ADBench resamples datasets < 1,000 samples with replacement (train/test leakage)

def adbench_split(X, y, seed, test_size=0.3, n_min=None, n_max=10000):
    '''Exact replica of ADBench DataGenerator.generator (generate_duplicates=True, minmax=True).'''
    n_min = (1000 if RESAMPLE_SMALL else 0) if n_min is None else n_min
    np.random.seed(seed); random.seed(seed)
    if len(y) < n_min:
        np.random.seed(seed); random.seed(seed)
        idx = np.random.choice(np.arange(len(y)), n_min, replace=True); X, y = X[idx], y[idx]
    if len(y) > n_max:
        np.random.seed(seed); random.seed(seed)
        idx = np.random.choice(np.arange(len(y)), n_max, replace=False); X, y = X[idx], y[idx]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, shuffle=True, stratify=y)
    sc = MinMaxScaler().fit(X_tr)
    return sc.transform(X_tr), y_tr, sc.transform(X_te), y_te

from pyod.models.iforest import IForest
from pyod.models.cblof import CBLOF
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.pca import PCA
from pyod.models.hbos import HBOS
from pyod.models.knn import KNN
from pyod.models.ocsvm import OCSVM
from pyod.models.sod import SOD
from pyod.models.lof import LOF
from pyod.models.loda import LODA
from pyod.models.cof import COF
from pyod.models.deep_svdd import DeepSVDD
PYOD = {'IForest': IForest, 'CBLOF': CBLOF, 'ECOD': ECOD, 'COPOD': COPOD, 'PCA': PCA, 'HBOS': HBOS,
        'KNN': KNN, 'OCSVM': OCSVM, 'SOD': SOD, 'LOF': LOF, 'LODA': LODA, 'COF': COF}
METHODS = list(PYOD) + ['DeepSVDD', 'DAGMM']

def set_seed(seed):
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def fit_score(name, X_fit, X_test, seed):
    '''Default hyperparameters, as in ADBench. Returns (train scores, test scores).'''
    set_seed(seed)
    if name in PYOD:
        m = PYOD[name]().fit(X_fit)
        return m.decision_scores_, m.decision_function(X_test)
    if name == 'DeepSVDD':
        kw = {'n_features': X_fit.shape[1]} if 'n_features' in inspect.signature(DeepSVDD).parameters else {}
        m = DeepSVDD(**kw).fit(X_fit)
        return m.decision_scores_, m.decision_function(X_test)
    if name == 'DAGMM':
        from adbench.baseline.DAGMM.run import DAGMM
        m = DAGMM(seed=seed).fit(X_fit, y_train=np.zeros(len(X_fit)))
        return m.predict_score(X_fit, X_fit), m.predict_score(X_fit, X_test)
    raise ValueError(name)
print('Methods:', METHODS)

In [ ]:
# ── One-time cleanup: drop small-dataset results made with resampling ──
SMALL = ['4_breastw', '14_glass', '15_Hepatitis', '18_Ionosphere', '21_Lymphography', '29_Pima',
         '37_Stamps', '39_vertebral', '42_WBC', '43_WDBC', '45_wine', '46_WPBC']
marker = os.path.join(DRIVE_ROOT, '.small_rerun_baselines')
if not RESAMPLE_SMALL and not os.path.exists(marker):
    for f in os.listdir(OUT_DIR):
        if f.split('__')[0] in SMALL: os.remove(os.path.join(OUT_DIR, f))
    open(marker, 'w').write('done'); print('small-dataset baseline results cleared')

# ── Main loop (resumable; one file per protocol, dataset and seed) ─────
def out_path(ds, protocol, seed): return os.path.join(OUT_DIR, f'{ds}__{protocol}__seed{seed}.npz')

order = sorted(ADBENCH_47, key=lambda n: os.path.getsize(os.path.join(DATA_DIR, n + '.npz')))
jobs = [(p, d, s) for d in order for s in SEEDS for p in PROTOCOLS if not os.path.exists(out_path(d, p, s))]
print(len(jobs), 'jobs to run')
for i, (protocol, ds, seed) in enumerate(jobs):
    t0 = time.time()
    raw = np.load(os.path.join(DATA_DIR, ds + '.npz'), allow_pickle=True)
    X_tr, y_tr, X_te, y_te = adbench_split(raw['X'].astype(float), raw['y'].astype(int), seed)
    X_fit = X_tr if protocol == 'adbench' else X_tr[y_tr == 0]
    res = {'y_test': y_te.astype(np.int8), 'n_train_anomaly': int(y_tr.sum()) if protocol == 'adbench' else 0}
    errors = {}
    for name in METHODS:
        try:
            tr, te = fit_score(name, X_fit, X_te, seed)
            res[f'{name}__train'] = np.asarray(tr, np.float64); res[f'{name}__test'] = np.asarray(te, np.float64)
        except Exception as e:
            errors[name] = repr(e)[:200]
    res['errors'] = json.dumps(errors)
    np.savez_compressed(out_path(ds, protocol, seed), **res)
    print(f'[{i+1}/{len(jobs)}] {ds} {protocol} seed{seed}: {time.time()-t0:.0f}s  failed={list(errors)}')
print('Finished.')

In [ ]:
# ── Summary: mean AUROC / AP per method and protocol ───────────────────
def clean(s):
    s = np.asarray(s, float).copy(); f = np.isfinite(s); hi, lo = (s[f].max(), s[f].min()) if f.any() else (0, 0)
    s[np.isposinf(s)] = hi + 1; s[np.isneginf(s) | np.isnan(s)] = lo - 1; return s
rows = []
for f in sorted(os.listdir(OUT_DIR)):
    ds, protocol, seed = f[:-4].split('__'); z = np.load(os.path.join(OUT_DIR, f)); y = z['y_test']
    for m in METHODS:
        if f'{m}__test' in z.files:
            s = clean(z[f'{m}__test'])
            rows.append({'protocol': protocol, 'dataset': ds, 'method': m, 'seed': seed,
                         'auroc': roc_auc_score(y, s), 'ap': average_precision_score(y, s)})
res = pd.DataFrame(rows)
res.to_csv(os.path.join(DRIVE_ROOT, 'baselines_per_seed.csv'), index=False)
print((100 * res.pivot_table(index='method', columns='protocol', values='auroc')).round(1)
      .sort_values('adbench', ascending=False).to_string())